<div style='background-color:#002147; padding:20px; border-radius:8px'>
<h1 style='color:white;'>Cálculo Numérico</h1>
<h3 style='color:white;'>Resolução de sistemas lineares: métodos iterativos</h3>
</div>

### 🎯 Objetivos de aprendizagem

- construir a forma iterativa 
$
\boldsymbol{x}^{(k+1)} = C\,\boldsymbol{x}^{(k)} + \boldsymbol{g}
$;

- implementar o método de **Gauss-Jacobi**;
- verificar a convergência pelo **critério das linhas**;
- implementar o método de **Gauss-Seidel**;
- reordenar as equações do sistema (troca de linhas), quando necessário, para favorecer a convergência.

## $ \S 0 $ Preparação do ambiente

Execute a célula abaixo para carregar as bibliotecas utilizadas neste Jupyter Notebook. 💻

Usaremos `numpy` para manipular matrizes e vetores ao longo das atividades.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

**🔹 Funções auxiliares**

As funções a seguir serão utilizadas para exibir as iterações, verificar critérios de convergência e resolver sistemas pelos métodos iterativos. Após executá-las, você pode minimizar esta célula.

In [ ]:
def animar_trajetoria_2x2(A, b, traj, titulo="Trajetória iterativa", interval=900):
    """
    Cria uma animação da trajetória iterativa para sistemas 2x2.

    As setas pontilhadas são criadas antes da animação e ficam invisíveis.
    Em cada quadro, ativamos apenas as setas já percorridas. Esse formato
    costuma funcionar melhor no HTML gerado pelo Jupyter.
    """
    from matplotlib.patches import FancyArrowPatch

    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    traj = np.asarray(traj, dtype=float)
    sol = np.linalg.solve(A, b)

    pontos = np.vstack([traj, sol])
    xmin, xmax = pontos[:, 0].min(), pontos[:, 0].max()
    ymin, ymax = pontos[:, 1].min(), pontos[:, 1].max()
    dx = xmax - xmin if xmax != xmin else 1.0
    dy = ymax - ymin if ymax != ymin else 1.0
    margem = 0.35

    xlim = (xmin - margem * dx, xmax + margem * dx)
    ylim = (ymin - margem * dy, ymax + margem * dy)
    xs = np.linspace(xlim[0], xlim[1], 300)

    fig, ax = plt.subplots(figsize=(7, 5))

    for i, cor in enumerate(["#1f77b4", "#b22222"]):
        ys = (b[i] - A[i, 0] * xs) / A[i, 1]
        ax.plot(xs, ys, color=cor, linewidth=2, label=f"equação {i+1}")

    ax.scatter([sol[0]], [sol[1]], color="purple", marker="*", s=140, label="solução", zorder=5)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(titulo)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend(loc="best")

    # Pontos aparecem separadamente das setas.
    pontos_iter, = ax.plot([], [], "ko", markersize=5, zorder=6)
    ponto_atual, = ax.plot([], [], "o", color="green", markersize=8, zorder=7)
    texto = ax.text(xlim[0], ylim[1], "", color="#222222", va="top")

    # Criamos todas as setas uma vez só. Depois a animação só muda a visibilidade.
    setas = []
    for i in range(len(traj) - 1):
        seta = FancyArrowPatch(
            (traj[i, 0], traj[i, 1]),
            (traj[i + 1, 0], traj[i + 1, 1]),
            arrowstyle="->",
            mutation_scale=18,
            linewidth=1.8,
            linestyle=(0, (1.2, 2.2)),
            color="#111111",
            shrinkA=5,
            shrinkB=5,
            zorder=4,
        )
        seta.set_visible(False)
        ax.add_patch(seta)
        setas.append(seta)

    def init():
        pontos_iter.set_data([], [])
        ponto_atual.set_data([], [])
        texto.set_text("")
        for seta in setas:
            seta.set_visible(False)
        return [pontos_iter, ponto_atual, texto] + setas

    def update(k):
        dados = traj[:k + 1]
        pontos_iter.set_data(dados[:, 0], dados[:, 1])
        ponto_atual.set_data([traj[k, 0]], [traj[k, 1]])

        for i, seta in enumerate(setas):
            seta.set_visible(i < k)

        texto.set_text(f"k = {k}\nx = ({traj[k, 0]:.6f}, {traj[k, 1]:.6f})")
        return [pontos_iter, ponto_atual, texto] + setas

    ani = FuncAnimation(
        fig,
        update,
        frames=len(traj),
        init_func=init,
        interval=interval,
        blit=False,
    )

    plt.close(fig)
    return HTML(ani.to_jshtml())

def norma_inf(v):
    return np.linalg.norm(v, np.inf)

def norma_1(v):
    return np.linalg.norm(v, 1)

def norma_2(v):
    return np.linalg.norm(v, 2)

def erro_absoluto(x_aprox, x_ref, norma=norma_inf):
    return norma(np.asarray(x_aprox, dtype=float) - np.asarray(x_ref, dtype=float))

def erro_relativo(x_aprox, x_ref, norma=norma_inf):
    x_ref = np.asarray(x_ref, dtype=float)
    denom = norma(x_ref)
    if np.isclose(denom, 0.0):
        raise ValueError("Erro relativo não definido quando a referência tem norma zero.")
    return erro_absoluto(x_aprox, x_ref, norma=norma) / denom

def validar_sistema(A, b, x0):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    x0 = np.asarray(x0, dtype=float)

    n = A.shape[0]

    if A.shape != (n, n):
        raise ValueError("A deve ser uma matriz quadrada.")

    if b.shape != (n,):
        raise ValueError("b deve ter dimensão compatível com A.")

    if x0.shape != (n,):
        raise ValueError("x0 deve ter dimensão compatível com A.")

    if np.any(np.isclose(np.diag(A), 0.0)):
        raise ValueError("Os elementos da diagonal de A devem ser não nulos.")
        
    return A, b, x0

def imprimir_iteracoes(historico):
    historico = np.asarray(historico, dtype=float)
    n = historico.shape[1]

    cabecalho = (
        " k | "
        + " ".join([f"x{i+1}".rjust(12) for i in range(n)])
        + " |   ||Δx||∞   |   ||Δx||∞ / ||x^(k)||∞"
    )
    print(cabecalho)
    print("-" * len(cabecalho))

    for k, x in enumerate(historico):
        valores = " ".join([f"{v:12.6f}" for v in x])

        if k == 0:
            erro_abs_str = " " * 10
            erro_rel_str = " " * 10
        else:
            x_ant = historico[k-1]

            erro_abs = erro_absoluto(x, x_ant, norma=norma_inf)
            denom = norma_inf(x)
            erro_rel = erro_abs / denom if denom != 0 else erro_abs

            erro_abs_str = f"{erro_abs:10.3e}"
            erro_rel_str = f"{erro_rel:10.3e}"

        print(f"{k:2d} | {valores} | {erro_abs_str} | {erro_rel_str}")

<div style="background-color: #fff3b0; padding: 12px; border-radius: 6px;">

⚠️ **Atenção:** em Python, os índices começam em 0. Portanto, `x[0]` representa $x_1$, `x[1]` representa $x_2$ e assim por diante.

</div>

## $ \S 1 $ Ideia geral dos métodos iterativos

A ideia central dos métodos iterativos é generalizar o método do **ponto fixo** para sistemas lineares.

Considere o sistema linear da forma
$\boldsymbol{A}\boldsymbol{x} = \boldsymbol{b}$. Esse sistema pode ser reescrito na forma

$$
\boldsymbol{x} = C\,\boldsymbol{x} + \boldsymbol{g},
$$

ou, de forma equivalente,

$$
\boldsymbol{x} = \phi(\boldsymbol{x}),
$$

onde $C \in \mathbb{R}^{n \times n}$, $\boldsymbol{g} \in \mathbb{R}^n$ e
$\phi : \mathbb{R}^n \to \mathbb{R}^n$ é uma função associada ao método.

A solução do sistema passa, então, a ser interpretada como um **ponto fixo**
da função $\phi$.

A partir de uma aproximação inicial $\boldsymbol{x}^{(0)}$, constrói-se uma sequência
de vetores dada por

$$
\boldsymbol{x}^{(k+1)} = C\,\boldsymbol{x}^{(k)} + \boldsymbol{g}
= \phi\big(\boldsymbol{x}^{(k)}\big), \quad k = 0,1,2,\ldots
$$

🎯 A sequência de aproximações $\{\boldsymbol{x}^{(k)}\}_{k=0}^\infty$ converge para a solução exata $\boldsymbol{x}^*$ do sistema linear, desde que as condições de convergência sejam satisfeitas.

## $ \S 2 $ Normas, erro e critérios de parada

Nos métodos iterativos, é necessário medir o tamanho de vetores como

$$
\boldsymbol{x}^{(k)} - \boldsymbol{x}^{(k-1)}, \qquad
\boldsymbol{x}^{(k)} - \boldsymbol{x}^*, \qquad
\boldsymbol{A}\boldsymbol{x}^{(k)} - \boldsymbol{b}.
$$

Para isso, utilizamos **normas vetoriais**. As mais comuns em Cálculo Numérico são:

$$
\text{Norma 1: }\|\boldsymbol{x}\|_1 = \sum_{i=1}^n |x_i|,
\qquad
\text{Norma 2: }\|\boldsymbol{x}\|_2 = \left( \sum_{i=1}^n x_i^2 \right)^{1/2},
\qquad
\text{Norma $\infty$: }\|\boldsymbol{x}\|_\infty = \max_{1 \le i \le n} |x_i|.
$$

<div style="border:2px solid #444; padding:15px; border-radius:8px; background:#f7f7f7">

**Definição: erro absoluto e erro relativo**

Se $\boldsymbol{x}$ é a solução exata e $\bar{\boldsymbol{x}}$ é uma aproximação, então:

$$
E_{abs} = \|\boldsymbol{x} - \bar{\boldsymbol{x}}\|,
\qquad
E_{rel} = \frac{\|\boldsymbol{x} - \bar{\boldsymbol{x}}\|}{\|\boldsymbol{x}\|},
\quad \boldsymbol{x} \neq \boldsymbol{0}.
$$

</div>

O erro relativo é particularmente útil, pois mede o erro em proporção ao tamanho da solução.

Na prática, como a solução exata $\boldsymbol{x}^*$ geralmente é desconhecida, utilizam-se **critérios de parada computáveis**, como:

$$
\|\boldsymbol{x}^{(k)} - \boldsymbol{x}^{(k-1)}\|_\infty < \varepsilon,
$$

$$
\frac{\|\boldsymbol{x}^{(k)} - \boldsymbol{x}^{(k-1)}\|_\infty}{\|\boldsymbol{x}^{(k)}\|_\infty} < \varepsilon,
$$

ou ainda o **resíduo relativo**

$$
\frac{\|\boldsymbol{A}\boldsymbol{x}^{(k)} - \boldsymbol{b}\|_\infty}{\|\boldsymbol{b}\|_\infty} < \varepsilon.
$$

## §3 Motivação geométrica no caso $2\times2$

Vamos entender a ideia dos métodos iterativos a partir de uma interpretação geométrica simples.

No caso de duas equações e duas incógnitas, cada equação representa uma **reta no plano**. A solução do sistema corresponde ao **ponto de interseção dessas retas**.

Considere o sistema:

$$
\begin{cases}
-7.50x + 7.30y = -4.65,\\
-2.80x + 9.00y = 49.8.
\end{cases}
$$

Cada equação define uma reta. O objetivo é encontrar o ponto onde essas duas retas se cruzam.

Isolando as variáveis na forma usada pelo método de Jacobi, obtemos

$$
x = \frac{4.65 + 7.30y}{7.50},
\qquad
y = \frac{49.8 + 2.80x}{9.00}.
$$

Essas expressões permitem construir aproximações sucessivas para a solução.

### §3.1 Método de Jacobi

No método de Jacobi, cada nova aproximação é calculada **usando apenas os valores da iteração anterior**:

$$
x^{(k+1)} = \frac{4.65 + 7.30y^{(k)}}{7.50},
\qquad
y^{(k+1)} = \frac{49.8 + 2.80x^{(k)}}{9.00}.
$$

🔹 **Interpretação geométrica**

Geometricamente, cada passo do método utiliza as retas do sistema para corrigir as coordenadas da aproximação atual:

- a primeira equação determina a nova coordenada $x$, fixando $y = y^{(k)}$;
- a segunda equação determina a nova coordenada $y$, utilizando $x = x^{(k)}$.

Assim, cada iteração corresponde a movimentos alternados entre as duas retas.


🔹 **Conclusão**

A sequência gerada converge para o ponto de interseção das retas, que corresponde à solução do sistema.

Para visualizar esse processo, podemos implementar o método de Jacobi e acompanhar a trajetória das aproximações no plano.
Veja o código abaixo:

In [ ]:
def jacobi_traj_2x2(A, b, x0, n_iter):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)

    # chute inicial
    x, y = float(x0[0]), float(x0[1])

    traj = [(x, y)]

    # validação
    if A.shape != (2, 2):
        raise ValueError("A matriz deve ser 2x2.")
    if A[0,0] == 0 or A[1,1] == 0:
        raise ValueError("Elementos diagonais não podem ser zero.")

    for _ in range(n_iter):
        # Jacobi: usa valores antigos
        x_novo = (b[0] - A[0,1]*y) / A[0,0]
        y_novo = (b[1] - A[1,0]*x) / A[1,1]

        x, y = x_novo, y_novo
        traj.append((x, y))

    return np.array(traj)

🧪[__Exemplo 3-03.1__](#exemplo-3-03.1)

Visualize as primeiras iterações do método de Jacobi iniciando em $(50,\,40)$.

In [ ]:
A_geo = np.array([[-7.50, 7.30],
                  [-2.80, 9.00]])

b_geo = np.array([-4.65, 49.8])

x0_geo = np.array([50.0, 40.0])


traj_j = jacobi_traj_2x2(A_geo, b_geo, x0_geo, n_iter=6)
imprimir_iteracoes(traj_j)

animar_trajetoria_2x2(A_geo, b_geo, traj_j, titulo="Jacobi no plano")

### §3.2 Método de Gauss-Seidel

No método de Gauss-Seidel, as atualizações são feitas **sequencialmente**, utilizando imediatamente os valores mais recentes:

$$
x^{(k+1)} = \frac{4.65 + 7.30y^{(k)}}{7.50},
\qquad
y^{(k+1)} = \frac{49.8 + 2.80x^{(k+1)}}{9.00}.
$$

🔹 **Interpretação geométrica**

Dado um ponto $(x^{(k)}, y^{(k)})$:

1. Calcula-se $x^{(k+1)}$ na primeira reta, mantendo $y = y^{(k)}$;
2. Em seguida, calcula-se $y^{(k+1)}$ na segunda reta, **já usando $x^{(k+1)}$**.

Assim, cada iteração corresponde a um movimento em duas etapas, utilizando imediatamente a informação mais recente.

🔹 **Conclusão**

A sequência gerada tende ao ponto de interseção das retas, que corresponde à solução do sistema, geralmente com convergência mais rápida do que no método de Jacobi.

Veja o código abaixo e analise a diferença entre os dois métodos.

In [ ]:
def gauss_seidel_traj_2x2(A, b, x0, n_iter):
    x = np.asarray(x0, dtype=float).copy()
    traj = [x.copy()]

    for _ in range(n_iter):
        x[0] = (b[0] - A[0, 1] * x[1]) / A[0, 0]
        x[1] = (b[1] - A[1, 0] * x[0]) / A[1, 1]
        traj.append(x.copy())

    return np.array(traj)

🧪[__Exemplo 3-03.2__](#exemplo-3-03.2)

Visualize as primeiras iterações do método de Gauss-Seidel iniciando em $(50,\,40)$.

In [ ]:
A_geo = np.array([[-7.50, 7.30],
                  [-2.80, 9.00]])

b_geo = np.array([-4.65, 49.8])

x0_geo = np.array([50.0, 40.0])

traj_gs = gauss_seidel_traj_2x2(A_geo, b_geo, x0_geo, 6)

imprimir_iteracoes(traj_gs)

animar_trajetoria_2x2(A_geo, b_geo, traj_gs, titulo="Gauss-Seidel no plano")

## $ \S 4 $ Descrição dos métodos iterativos para sistemas lineares

### $ \S 4.1 $ Método de Gauss-Jacobi
Considere o sistema linear

$$
\begin{cases}
a_{11}x_1 + a_{12}x_2 + \cdots + a_{1n}x_n = b_1 \\
a_{21}x_1 + a_{22}x_2 + \cdots + a_{2n}x_n = b_2 \\
\vdots \\
a_{n1}x_1 + a_{n2}x_2 + \cdots + a_{nn}x_n = b_n
\end{cases}
$$

Supondo $a_{ii} \neq 0, \; i=1,2,\ldots,n$, isolamos o vetor $\boldsymbol{x}$ mediante a separação da diagonal da matriz $A$:

$$
\begin{cases}
x_1 = \dfrac{1}{a_{11}}\left(b_1 - a_{12}x_2 - \cdots - a_{1n}x_n\right) \\
x_2 = \dfrac{1}{a_{22}}\left(b_2 - a_{21}x_1 - \cdots - a_{2n}x_n\right) \\
\vdots \\
x_n = \dfrac{1}{a_{nn}}\left(b_n - a_{n1}x_1 - \cdots - a_{n,n-1}x_{n-1}\right)
\end{cases}
$$

O método de Jacobi consiste em gerar uma sequência de aproximações
$\boldsymbol{x}^{(k)}$ a partir de um chute inicial $\boldsymbol{x}^{(0)}$, utilizando

$$
x_i^{(k+1)} =
\frac{1}{a_{ii}}\left(b_i - \sum_{\substack{j=1 \\ j \ne i}}^{n} a_{ij}x_j^{(k)}\right),
\quad i = 1,2,\ldots,n.
$$

📌 **Observação**

No método de Gauss-Jacobi, todos os componentes de $\boldsymbol{x}^{(k+1)}$ são calculados
utilizando exclusivamente os valores da iteração anterior $\boldsymbol{x}^{(k)}$.
Assim, as atualizações são realizadas de forma **simultânea**.

<div style="border:2px solid #1f77b4; padding:12px; border-radius:8px; background:#eef6fc">

📘 **Nota histórica**

O método foi publicado pelo matemático alemão **Carl Gustav Jacob Jacobi**, em 1845, sendo uma de suas diversas contribuições à Matemática.

Apesar disso, o método também é frequentemente associado a **Gauss**, embora não haja um consenso claro sobre quando ou por que seu nome passou a ser vinculado a esse procedimento.

</div>

In [ ]:
def jacobi(A, b, x0, eps, max_iter, verbose=True):
    A, b, x = validar_sistema(A, b, x0)
    n = len(b)
    historico = [x.copy()]
    convergiu = False

    for k in range(1, max_iter + 1):
        x_novo = np.zeros(n)

        for i in range(n):
            soma = 0.0
            for j in range(n):
                if j != i:
                    soma += A[i, j] * x[j]
            x_novo[i] = (b[i] - soma) / A[i, i]

        historico.append(x_novo.copy())

        # 🔹 erro absoluto
        erro_abs = norma_inf(x_novo - x)

        # 🔹 erro relativo
        denom = norma_inf(x_novo)
        erro_rel = erro_abs / denom if denom != 0 else erro_abs

        x = x_novo

        # 🔹 critério de parada
        if erro_abs < eps or erro_rel < eps:
            convergiu = True
            break

    historico = np.array(historico)

    # 🔹 impressão controlada
    if verbose:
        print("\n🔹 Iterações (Jacobi):")
        imprimir_iteracoes(historico)

        print("\n🔹 Resumo final:")
        print("Iterações:", k)
        print("Erro absoluto:", erro_abs)
        print("Erro relativo:", erro_rel)
        print("Resíduo:", norma_inf(A @ x - b))
        print("Convergiu:", convergiu)    

    return x

📝[__Exercício 3-03.1__](#exercicio-3-03.1): Resolva o sistema linear

$$
\begin{cases}
10x_1+2x_2+x_3=7,\\
x_1+5x_2+x_3=-8,\\
2x_1+3x_2+10x_3=6,
\end{cases}
$$

pelo método de Gauss-Jacobi com $x^{(0)}=(0.7,\,-1.6,\,0.6)^T$ e $\varepsilon=0.05$.

### §4.2 Método iterativo de Gauss-Seidel

No método de Gauss-Seidel, isolamos cada variável como no método de Jacobi, mas com uma diferença fundamental: **os valores atualizados são utilizados imediatamente dentro da mesma iteração**.

Assim, as equações iterativas ficam:

$$
\begin{cases}
x_1^{(k+1)} = \dfrac{1}{a_{11}}
\left(b_1 - a_{12}x_2^{(k)} - \cdots - a_{1n}x_n^{(k)}\right), \\[6pt]

x_2^{(k+1)} = \dfrac{1}{a_{22}}
\left(b_2 - a_{21}x_1^{(k+1)} - \cdots - a_{2n}x_n^{(k)}\right), \\[6pt]

\vdots \\[6pt]

x_n^{(k+1)} = \dfrac{1}{a_{nn}}
\left(b_n - a_{n1}x_1^{(k+1)} - \cdots - a_{n,n-1}x_{n-1}^{(k+1)}\right).
\end{cases}
$$

onde assumimos que $a_{11}, \ldots, a_{nn} \neq 0$.

De forma compacta:

$$
\boxed{
x_i^{(k+1)}=
\frac{1}{a_{ii}}\left(
 b_i
 -\sum_{j<i}a_{ij}x_j^{(k+1)}
 -\sum_{j>i}a_{ij}x_j^{(k)}
\right).
}
$$

<div style="border:2px solid #1f77b4; padding:12px; border-radius:8px; background:#eef6ff">

💡 **Diferença entre Jacobi e Gauss-Seidel**

A principal diferença entre os métodos está na forma como os valores são atualizados em cada iteração:

- **Jacobi:** calcula todos os valores de $x^{(k+1)}$ usando apenas os valores da iteração anterior $x^{(k)}$;
- **Gauss-Seidel:** atualiza cada componente imediatamente, utilizando os valores mais recentes já calculados na mesma iteração.

📌 Em outras palavras, o método de Gauss-Seidel “aproveita” as informações mais novas ao longo da iteração, o que geralmente resulta em **convergência mais rápida** em comparação ao método de Jacobi.

</div>

In [ ]:
def gauss_seidel(A, b, x0, eps, max_iter, verbose=True):
    A, b, x = validar_sistema(A, b, x0)
    n = len(b)
    historico = [x.copy()]
    convergiu = False

    for k in range(1, max_iter + 1):
        x_antigo = x.copy()

        for i in range(n):
            soma = 0.0
            for j in range(n):
                if j != i:
                    soma += A[i, j] * x[j]
            x[i] = (b[i] - soma) / A[i, i]

        historico.append(x.copy())

        # 🔹 erro absoluto
        erro_abs = norma_inf(x - x_antigo)

        # 🔹 erro relativo (forma teórica)
        denom = norma_inf(x)
        erro_rel = erro_abs / denom if denom != 0 else erro_abs

        # 🔹 critério de parada
        if erro_abs < eps or erro_rel < eps:
            convergiu = True
            break

    historico = np.array(historico)

    # 🔹 impressão controlada
    if verbose:
        print("\n🔹 Iterações (Gauss-Seidel):")
        imprimir_iteracoes(historico)

        print("\n🔹 Resumo final:")
        print("Iterações:", k)
        print("Erro absoluto:", erro_abs)
        print("Erro relativo:", erro_rel)
        print("Resíduo:", norma_inf(A @ x - b))
        print("Convergiu:", convergiu)    
        
    return x

📝[__Exercício 3-03.2__](#exercicio-3-03.2): Resolva o sistema linear

$$
\begin{cases}
5x_1+x_2+x_3=5,\\
3x_1+4x_2+x_3=6,\\
3x_1+3x_2+6x_3=0,
\end{cases}
$$

pelo método de Gauss-Seidel com $x^{(0)}=(0,0,0)^T$ e $\varepsilon=5\times 10^{-2}$.

_Solução:_

### $ \S 3.3 $ Um critério de convergência

Antes de apresentar os critérios de convergência, vamos analisar, na prática, como os métodos iterativos podem se comportar a partir do exercício a seguir.

📝[__Exercício 3-03.3__](#exercicio-3-03.3): Considere o sistema com as equações trocadas:

$$
\begin{cases}
-2.80x + 9.00y = 49.8,\\
-7.50x + 7.30y = -4.65.
\end{cases}
$$

- (a) Utilize a função `jacobi_traj_2x2` para gerar a trajetória do método de Jacobi,
realizando **4 iterações**, considerando o ponto inicial $(x^{(0)}, y^{(0)}) = (50,\;40).$

- (b) Descreva o comportamento do método:

    - O método **converge ou diverge**?
    - O que acontece com as iterações ao longo do tempo?


📌 **Observação**

Compare com o [__Exemplo 3-03.1__](#exemplo-3-03.1)  e explique por que a troca da ordem das equações altera o comportamento do método de Jacobi.

_Solução:_

<div style="border:2px solid #d62728; padding:12px; border-radius:8px; background:#fff2f2">

⚠️ **Observação importante**

- O que observamos acima é um pouco surpreendente. A sequência gerada pelo método de Gauss-Jacobi agora se afasta da solução. Isso ocorreu devido à simples troca da ordem das equações, o que mostra que o método é bastante sensível.

💡 **Boa notícia**

- Existem critérios que estabelecem **condições suficientes para a convergência** de métodos iterativos, sem a necessidade de executar as iterações.  

</div>

<div style="border:2px solid #444; padding:15px; border-radius:8px; background:#f7f7f7">

**Teorema 1 (Critério das linhas)**

Seja o sistema linear $\boldsymbol{A}\boldsymbol{x} = \boldsymbol{b}$ e seja

$$
\alpha_k = \frac{\sum_{\substack{j=1 \\ j \ne k}}^{n} |a_{kj}|}{|a_{kk}|}, \quad k = 1,2,\ldots,n.
$$

Se

$$
\alpha = \max_{1 \le k \le n} \{ \alpha_k \} < 1,
$$

então os métodos de **Gauss-Jacobi** e **Gauss-Seidel** geram uma sequência $\{\boldsymbol{x}^{(k)}\}$ convergente para a solução do sistema dado, independentemente da escolha da aproximação inicial $\boldsymbol{x}^{(0)}$.

</div>
<div style="border:2px solid #ff9800; padding:12px; border-radius:8px; background:#fff8e1">

⚠️ **Atenção**

Esse critério é **suficiente, mas não necessário**: se ele falhar, o método ainda pode convergir.

</div>

📌 **Sobre o código**

O código em Python a seguir implementa a verificação do **critério das linhas**.

In [ ]:
def criterio_linhas(A, verbose=True):
    A = np.asarray(A, dtype=float)
    n = A.shape[0]
    alphas = np.zeros(n)

    for k in range(n):
        if A[k, k] == 0:
            raise ValueError(f"Elemento diagonal A[{k},{k}] = 0")

        soma = np.sum(np.abs(A[k, :])) - abs(A[k, k])
        alphas[k] = soma / abs(A[k, k])

    alpha = np.max(alphas)

    if verbose:
        print("\n🔎 Critério das linhas:")
        print("alpha_k =", alphas)
        print("alpha =", alpha)

        if alpha < 1:
            print("✅ Satisfaz o critério das linhas (converge)")
        else:
            print("⚠️ Critério não satisfeito (pode convergir ou não)")

    return alpha, alphas

🧪[__Exemplo 3-03.3__](#exemplo-3-03.3) Resolva o sistema linear

$$
\begin{cases}
x_1+x_2=3,\\
x_1-3x_2=-3,
\end{cases}
$$

use $x^{(0)}=(0.5,0.6)^T$, $\varepsilon=0.01$. Verifique que o método de Jacobi converge para a solução exata, mesmo que o critério das linhas não seja satisfeito. Que conclusão você pode tirar sobre a relação entre o critério das linhas e a convergência do método de Jacobi?

In [ ]:
A = np.array([[1.0, 1.0],
              [1.0, -3.0]])
b = np.array([3.0, -3.0])

x0 = np.array([0.5, 0.6])
eps = 0.01
max_iter = 100

# 🔹 Critério das linhas
alpha, alphas = criterio_linhas(A, verbose=True)

x = jacobi(A, b, x0, eps, max_iter, verbose=True)
print("\nx0 =", x0)
print("solução aproximada:", x)

print("resíduo:", norma_inf(A @ x - b))

print("\nSolução exata:", np.linalg.solve(A, b))

Vimos que o critério das linhas é suficiente, mas não necessário. O método de Jacobi pode convergir mesmo quando o critério das linhas falha, como neste exemplo. Portanto, a ausência de satisfação do critério das linhas não implica necessariamente na divergência do método.

📝 [__Exercício 3-03.4__](#exercicio-3-03.4)

Utilize agora o método de **Gauss-Seidel** no mesmo sistema linear do [__Exemplo 3-03.3__](#exemplo-3-03.3) e verifique se o método converge.

_Solução:_

📝[__Exercício 3-03.2__](#exercicio-3-03.2) Resolva pelo método de Gauss-Jacobi, com $\varepsilon=10^{-3}$, o sistema

$$
A=\begin{bmatrix}
2 & 1\\
5 & 7
\end{bmatrix},
\qquad
b=\begin{bmatrix}
11\\13
\end{bmatrix}.
$$

Compare o número de iterações usando três aproximações iniciais diferentes:

$$
(7,-3)^T,\qquad (0,0)^T,\qquad (2000,-3000)^T.
$$

_Solução:_

📝[__Exercício 3-03.3__](#exercicio-3-03.3) Resolva pelo método de Gauss-Seidel, com $\varepsilon=10^{-3}$, o sistema

$$
A=\begin{bmatrix}
2 & 1\\
5 & 7
\end{bmatrix},
\qquad
b=\begin{bmatrix}
11\\13
\end{bmatrix}.
$$

Compare o número de iterações usando três aproximações iniciais diferentes:

$$
(7,-3)^T,\qquad (0,0)^T,\qquad (2000,-3000)^T.
$$
Verifique o comportamento do método de Gauss-Seidel para cada aproximação inicial e compare com o método de Jacobi.

_Solução:_

## $ \S 5 $ Trocas de linhas do sistema linear

Nem sempre um sistema linear satisfaz os critérios de convergência para os métodos iterativos. No entanto, é possível reordenar as equações do sistema (ou seja, trocar as linhas da matriz $A$ e do vetor $b$) para tentar satisfazer esses critérios.


<div style="background-color:#fff3b0; padding:12px; border-radius:6px;">

⚠️ **Observação importante:** reordenar as linhas do sistema não garante, sozinho, a convergência de Gauss-Jacobi ou Gauss-Seidel para qualquer sistema. Ele é uma estratégia para tentar obter uma forma equivalente mais favorável. A garantia vem quando a matriz reordenada satisfaz um critério de convergência.

</div>

📝[__Exercício 3-03.4__](#exercicio-3-03.4) A matriz abaixo não é diagonalmente dominante por linhas. Verifique se é possível reordenar suas linhas de modo que a matriz resultante satisfaça o **critério das linhas**. Caso seja possível, aplique o método de **Gauss-Seidel** ao sistema equivalente e verifique se o método converge.


$$
\begin{bmatrix}
1 & 4 & 1\\
1 & 1 & 3\\
5 & 1 & 1
\end{bmatrix}
\begin{bmatrix}
x_1\\x_2\\x_3
\end{bmatrix}
=
\begin{bmatrix}
6\\5\\7
\end{bmatrix}.
$$



_Solução:_

## $ \S 6 $ Exercícios

📝[__Exercício 3-03.4__](#exercicio-3-03.4): Aplique duas iterações do método de Jacobi ao sistema

$$
\begin{cases}
5x_1+x_2+x_3=5,\\
3x_1+4x_2+x_3=6,\\
3x_1+3x_2+6x_3=0,
\end{cases}
$$

começando com $x^{(0)}=(0,0,0)$.

📝[__Exercício 3-03.5__](#exercicio-3-03.5): Considere $x=(1,1)$ e $\bar x=(0{,}98,1{,}03)$.

(a) Calcule o erro absoluto usando as normas $\|\cdot\|_1$, $\|\cdot\|_2$ e $\|\cdot\|_\infty$.

(b) Calcule o erro relativo usando a norma infinito.

(c) Explique por que o erro relativo pode ser mais informativo que o erro absoluto.

📝[__Exercício 3-03.6__](#exercicio-3-03.6): 

(a) Aplique analítica e graficamente os métodos de Gauss-Jacobi e Gauss-Seidel no sistema abaixo

$$
\begin{cases}
2{.}5x-y=1{.}5,\\
x-2y=-1.
\end{cases}
$$

(b) Repita o item (a) para o sistema com as equações trocadas.

(c) Compare os resultados e explique a diferença de comportamento entre os dois sistemas.


📝[__Exercício 3-03.7__](#exercicio-3-03.7): Considere o sistema linear abaixo:
\begin{equation*}
\begin{cases}
& -x &+& 5y &+& z &=& 2 \\
& 2x &+& 3y &+& z &=& 3 \\
&  &+& 13y &+& 3z  &=& 6 
\end{cases}
\end{equation*}

(a) Mostre que ele não tem solução.

(b) O que acontece ao se tentar aplicar o método de Gauss-Seidel a ele?

📝[__Exercício 3-03.8__](#exercicio-3-03.8): Usando Gauss-Seidel, encontre uma aproximação com erro menor que $10^{-6}$ para o sistema

$$
\begin{bmatrix}
64 & -8 & 19 & -10 & 18 & -7 \\
4 & -41 & -25 & 3 & 0 & 4 \\
15 & -18 & 86 & 6 & -19 & -14 \\
-14 & 12 & 9 & -47 & -4 & -9 \\
5 & -15 & 12 & -13 & -48 & 2 \\
-2 & -15 & 7 & -3 & 1 & 30
\end{bmatrix}
x
=
\begin{bmatrix}
-7\\-11\\10\\-9\\2\\1
\end{bmatrix}.
$$

📝[__Exercício 3-03.9__](#exercicio-3-03.9): Considere o sistema

$$
\begin{bmatrix}
1 & 3 & 1\\
5 & 2 & 2\\
0 & 6 & 8
\end{bmatrix}
\begin{bmatrix}
x_1\\x_2\\x_3
\end{bmatrix}
=
\begin{bmatrix}
-2\\3\\-6
\end{bmatrix}.
$$

(a) Verifique que a matriz, na ordem apresentada, não é estritamente diagonalmente dominante por linhas.

(b) Faça uma troca de linhas que coloque, na diagonal, os coeficientes de maior módulo disponíveis para cada variável.

(c) Verifique se a matriz reordenada satisfaz o critério das linhas.

(d) Resolva o sistema reordenado usando Jacobi ou Gauss-Seidel.

### 🚀 [__Exercício 3-03.10__](#exercicio-3-03.10): Exercício Desafio

(a) Escreva um programa para resolver o seguinte sistema de $ n $ equações em $ n $ variáveis pelo método de Gauss-Seidel (o valor $ n $ deve ser especificado pelo usuário como entrada do programa):
\begin{equation*}
\begin{bmatrix}
2 & -1 & 0 & 0 & \cdots & 0 & 0 & 0 & 1 \\
-1 & 2 & -1 & 0 & \cdots & 0 & 0 & 0 & 0 \\
0 & -1 & 2 & -1 & \cdots & 0 & 0 & 0 & 0 \\
\vdots & \vdots & \vdots & \vdots & \cdots & \vdots & \vdots & \vdots \\
0 & 0 & 0 & 0 & \cdots & -1 & 2 & -1 & 0 \\
2 & -1 & 0 & 0 & \cdots & 0 & -1 & 2 & -1 \\
2 & -1 & 0 & 0 & \cdots & 0 & 0 & -1 & 2 \\
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2 \\
x_3 \\
\vdots \\
x_{n-2} \\
x_{n-1} \\
x_{n}
\end{bmatrix} =
\begin{bmatrix}
0 \\
0 \\
0 \\
\vdots \\
0 \\
0 \\
1
\end{bmatrix}.
\end{equation*}
*Dica:* Use as funções `tri` e `diag` da biblioteca `numpy` para construção de matrizes tridiagonais e diagonais, respectivamente.

(b) Execute seu programa com $ n = 20 $.

A solução exata é dada por $ x_i = -\frac{n}{4} + \frac{i}{2} $.

_Solução:_

---

## 📚 Referência

> 📘 RUGGIERO, Márcia A. Gomes; LOPES, Vera Lúcia da Rocha.  
> **Cálculo numérico: aspectos teóricos e computacionais**.  
> 2. ed. São Paulo: Makron Books, 1996.

> 🌐 Material complementar disponível em:  
> 👉 https://www.ime.unicamp.br/~biloti/an/211/index.html